# Cartographie Dynamique des Inondations

## Introduction
Les inondations sont des événements récurrents qui impactent lourdement les infrastructures et les populations en RDC. Ce notebook permet de délimiter précisément l'extension des eaux de surface en utilisant les indices spectraux d'absorption, facilitant ainsi la réponse d'urgence.

## Objectifs
*   **Identification de l'eau** : Isoler les pixels recouverts d'eau par rapport au sol sec.
*   **Cartographie d'extension** : Délimiter le contour des zones inondées.
*   **Calcul de surface** : Estimer l'étendue totale de la zone impactée.

## Méthodologie
1.  **Setup** : Installation des bibliothèques.
2.  **Acquisition** : Images Sentinel-2 filtrées par date (période de crue).
3.  **Analyse NDWI** : Application de l'indice de différence normalisée de l'eau.
4.  **Visualisation** : Carte binaire eau / terre.

In [ ]:
# ====================================================
# ÉTAPE 1 : Configuration
# ====================================================
!pip install geemap earthengine-api rasterio matplotlib -q

import ee, geemap, rasterio
import numpy as np
import matplotlib.pyplot as plt

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print("✅ Système prêt")

## Zone d'Étude (ROI)
Focus régional pour le monitoring hydrologique.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d\'étude')
Map

## Acquisition des Données
Nous exportons les bandes Vert (B3) et NIR (B8) nécessaires au calcul de l'eau.

In [ ]:
# ====================================================
# ÉTAPE 3 : Acquisition satellite
# ====================================================
image = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi).filterDate('2023-01-01', '2023-12-31')
         .median().clip(roi))

geemap.ee_export_image(image.select(['B3', 'B8']), 'flood.tif', scale=30, region=roi)

## Calcul NDWI et Masquage
L'indice NDWI exploite le fait que l'eau reflète le vert mais absorbe le NIR. Un score positif indique la présence d'eau.

In [ ]:
# ====================================================
# ÉTAPE 4 : Algorithme NDWI
# ====================================================
with rasterio.open('flood.tif') as src: bands = src.read().astype(np.float32)
green, nir = bands[0], bands[1]
ndwi = (green - nir) / (green + nir + 1e-8)

flood_mask = ndwi > 0.1

plt.figure(figsize=(10, 8))
plt.imshow(flood_mask, cmap='Blues')
plt.title("Extension des Inondations (NDWI)")
plt.axis('off')
plt.show()